# Cycling Management Chatbot (LionWeb)

This notebook provides a conversational interface for managing cycling teams and riders using natural language, powered by a LionWeb backend.

In [ ]:
# Setup and imports
import sys
sys.path.append('..')  # Add src to path

import ipywidgets as widgets
from IPython.display import display, HTML
from typing import Optional, List, Dict

# Import the NEW Cycling System
from src.cycling_system import CyclingSystem

class CyclingChatInterface:
    """Main chat interface for the cycling system."""
    
    def __init__(self, system):
        self.system = system
        self.conversation_state = {}
        self.active_clarification_widgets = {}
        self._create_widgets()
        self._setup_handlers()
        self._apply_styles()
    
    def _create_widgets(self):
        self.header = widgets.HTML(
            "<h1>Cycling Management Assistant 🚴</h1>"
            "<p>Manage teams and riders using natural language with a LionWeb backend.</p>"
        )
        available_models = self.system.get_available_models()
        self.model_selector = widgets.Dropdown(
            options=available_models,
            value=available_models[0] if available_models else None,
            description='Model:'
        )
        self.role_selector = widgets.RadioButtons(
            options=['admin', 'scheduler'],
            value='admin',
            description='Role:'
        )
        self.state_display = widgets.HTML()
        self.state_accordion = widgets.Accordion(
            children=[self.state_display],
            titles=('🚴 Model State',)
        )
        self.state_accordion.selected_index = None
        self.chat_history = widgets.VBox([])
        self.user_input = widgets.Textarea(
            placeholder='Type your request...', layout={'width': '95%', 'height': '80px'}
        )
        self.send_button = widgets.Button(description='Send', button_style='success', icon='paper-plane')
        self.clear_button = widgets.Button(description='Clear', button_style='warning', icon='trash')
        self.confirm_button = widgets.Button(description='✅ Confirm', button_style='success', layout={'visibility': 'hidden'})
        self.cancel_button = widgets.Button(description='❌ Cancel', button_style='danger', layout={'visibility': 'hidden'})
        self.clarification_form_area = widgets.VBox([])

    def _setup_handlers(self):
        self.send_button.on_click(self._on_send)
        self.clear_button.on_click(self._on_clear)
        self.confirm_button.on_click(self._on_confirm)
        self.cancel_button.on_click(self._on_cancel)

    def _on_send(self, b):
        query = self.user_input.value.strip()
        if not query: return
        self._process_query(query)

    def _process_query(self, query: str):
        self._add_message(f"<b>You:</b> {query}", 'user')
        self.user_input.value = ''
        
        result = self.system.process_query(
            query, self.role_selector.value, self.model_selector.value, self.conversation_state
        )
        
        self.conversation_state = result.get('new_state', self.conversation_state)
        
        if result['status'] == 'clarification_needed':
            if 'understanding_html' in result: self._add_message(result['understanding_html'])
            self._build_clarification_form(result.get('clarification_data', {}))
        elif result['status'] == 'confirmation_needed':
            if 'understanding_html' in result: self._add_message(result['understanding_html'])
            self._add_message(result['message'])
            self._show_confirmation_buttons(True)
        elif result['status'] == 'direct_execute':
            if 'understanding_html' in result: self._add_message(result['understanding_html'])
            self._on_confirm(None)
        else:
            self._add_message(f"<b>Assistant:</b><br>{result['message']}")
            self._reset_conversation()

    def _build_clarification_form(self, clarification_data: Dict):
        self._clear_clarification_form()
        self.active_clarification_widgets = {}
        message = clarification_data.get("message", "Please provide more information.")
        fields = clarification_data.get("form_fields", [])
        form_elements = [widgets.HTML(f"<div class='assistant-bubble'>{message}</div>")]
        for field in fields:
            label = widgets.Label(f"{field['label']}:")
            prompt = widgets.HTML(f"<i style='font-size: smaller;'>{field['prompt']}</i>")
            widget = None
            if field["type"] == "boolean": widget = widgets.Checkbox(value=False, indent=False)
            elif field["type"] == "number": widget = widgets.IntText(value=0)
            else: widget = widgets.Text(value="")
            self.active_clarification_widgets[field["name"]] = widget
            form_elements.append(widgets.VBox([label, prompt, widget]))
        submit_button = widgets.Button(description="Submit Details", button_style='info', icon='check')
        submit_button.on_click(self._on_submit_clarification)
        form_elements.append(submit_button)
        self.clarification_form_area.children = form_elements
        self._toggle_main_input(False)

    def _on_submit_clarification(self, b):
        query = "\n".join([f"{name}: {widget.value}" for name, widget in self.active_clarification_widgets.items()])
        self._add_message(f"<b>You (form submission):</b><br><pre>{query}</pre>", 'user')
        self._clear_clarification_form()
        self._toggle_main_input(True)
        self._process_query(query)

    def _clear_clarification_form(self):
        self.clarification_form_area.children = []
        self.active_clarification_widgets = {}

    def _toggle_main_input(self, enabled: bool):
        self.user_input.disabled = not enabled
        self.send_button.disabled = not enabled

    def _on_clear(self, b):
        self._reset_conversation()
        self._clear_clarification_form()
        self._toggle_main_input(True)
        welcome_msg = "<b>Assistant:</b><br>Hello! I can help you manage cycling teams and riders. What would you like to do?"
        self.chat_history.children = [widgets.HTML(f"<div class='chat-bubble assistant-bubble'>{welcome_msg}</div>")]
        self._show_confirmation_buttons(False)

    def _on_cancel(self, b):
        self._show_confirmation_buttons(False)
        self._add_message("<b>Assistant:</b> Task cancelled.", 'system')
        self._reset_conversation()

    def display(self):
        display(self.styles)
        controls = widgets.HBox([self.model_selector, self.role_selector])
        input_area = widgets.HBox([self.user_input, self.send_button, self.clear_button])
        confirm_area = widgets.HBox([self.confirm_button, self.cancel_button])
        layout = widgets.VBox([self.header, controls, widgets.HTML("<hr>"), self.state_accordion, self.chat_history, self.clarification_form_area, input_area, confirm_area])
        display(layout)
        self._on_clear(None)
        self._update_state_display()
        
    def _add_message(self, content: str, msg_type: str = 'assistant'):
        bubble_class = {'user': 'user-bubble', 'assistant': 'assistant-bubble', 'system': 'system-bubble'}.get(msg_type, 'assistant-bubble')
        html = f"<div class='chat-bubble {bubble_class}'>{content}</div>"
        self.chat_history.children = list(self.chat_history.children) + [widgets.HTML(html)]

    def _on_confirm(self, b):
        self._show_confirmation_buttons(False)
        self._add_message("<i>Executing...</i>", 'system')
        result = self.system.execute_task(self.role_selector.value, self.conversation_state)
        self.chat_history.children = self.chat_history.children[:-1]
        if result['status'] == 'success':
            self._add_message(f"<b>Assistant:</b> {result['message']}", 'system')
            if result.get('action_type') == 'query':
                self._display_query_results(result.get('results', []), self.conversation_state.get("task_details", {}).get("action"))
            else:
                self._update_state_display()
        else:
            self._add_message(f"<b>Assistant:</b> {result['message']}")
        self._reset_conversation()

    def _show_confirmation_buttons(self, show: bool):
        visibility = 'visible' if show else 'hidden'
        self.confirm_button.layout.visibility = visibility
        self.cancel_button.layout.visibility = visibility
        self._toggle_main_input(not show)

    def _reset_conversation(self):
        self.conversation_state = {}
        self._toggle_main_input(True)

    def _update_state_display(self):
        try:
            state = self.system.get_current_state()
            riders = state.get('riders', [])
            teams = state.get('teams', [])
            html = "<h4>Riders</h4><table border='1' style='width:100%;'><tr><th>Name</th><th>Age</th><th>Country</th></tr>"
            for rider in riders:
                html += f"<tr><td>{rider.get('name')}</td><td>{rider.get('age')}</td><td>{rider.get('country')}</td></tr>"
            if not riders: html += "<tr><td colspan='3'><i>No riders yet</i></td></tr>"
            html += "</table>"
            html += "<h4 style='margin-top:15px;'>Teams</h4><table border='1' style='width:100%;'><tr><th>Name</th><th>Country</th><th>Budget (M)</th><th>Founded</th></tr>"
            for team in teams:
                html += (f"<tr><td>{team.get('name')}</td><td>{team.get('country')}</td><td>{team.get('budget')}</td><td>{team.get('founded')}</td></tr>")
            if not teams: html += "<tr><td colspan='4'><i>No teams yet</i></td></tr>"
            html += "</table>"
            self.state_display.value = html
        except Exception as e:
            self.state_display.value = f"<p style='color:red;'>Error: {e}</p>"

    def _display_query_results(self, results: List[Dict], action: str):
        if not results: 
            self._add_message("No results found for your query.")
            return
        html = ""
        if action == "find_rider":
            html += "<h4>Rider Search Results</h4><table border='1' style='width:100%;'><tr><th>Name</th><th>Age</th><th>Country</th></tr>"
            for rider in results:
                html += f"<tr><td>{rider.get('name')}</td><td>{rider.get('age')}</td><td>{rider.get('country')}</td></tr>"
        elif action == "find_team":
            html += "<h4 style='margin-top:15px;'>Team Search Results</h4><table border='1' style='width:100%;'><tr><th>Name</th><th>Country</th><th>Budget (M)</th><th>Founded</th></tr>"
            for team in results:
                html += (f"<tr><td>{team.get('name')}</td><td>{team.get('country')}</td><td>{team.get('budget')}</td><td>{team.get('founded')}</td></tr>")
        html += "</table>"
        self._add_message(html, 'assistant')

    def _apply_styles(self):
        self.styles = HTML("<style>.chat-bubble{max-width:80%;padding:12px;border-radius:10px;margin:8px 0;line-height:1.5;}.user-bubble{background-color:#E3F2FD;margin-left:20%;border:1px solid #90CAF9;}.assistant-bubble{background-color:#F3E5F5;margin-right:20%;border:1px solid #CE93D8;}.system-bubble{background-color:#E8F5E9;margin:0 auto;width:60%;text-align:center;border:1px solid #A5D6A7;}</style>")

# Initialize system and interface
cycling_system = CyclingSystem()
interface = CyclingChatInterface(cycling_system)

# Display the interface
interface.display()
